# Data Analytics and Visualization in Data-Centric EngineeringThis workbook contains the worked solution for the workshop. Use it to check the method, the code, and the reasoning behind each step.The example uses three months of hourly wind turbine telemetry to show cleaning, visualisation, normalisation, and decision-making.The case study compares three turbines:- WT-01: healthy and stable- WT-02: failed in Month 1 and then flatlined- WT-03: healthy at first, then begins to fail in Month 3Before starting, run the mock telemetry generation script so that `wind_turbine_telemetry.csv` is in the same folder as this notebook.

## Step 0: Mock Data SetupRun the generator script first. It creates the dataset used throughout the exercises and includes the failure patterns discussed in the workshop.

## Section 1: Data Ingestion & Data Cleaning [15 mins]Start with the raw telemetry and decide what should be kept, cleaned, or set aside. In practice, missing values and flatlines are often part of the story rather than simple noise.A row with missing values might mean a failed sensor, a broken data path, or a machine that has stopped. If we remove those rows too early, we lose useful evidence.Keep the raw records, then create a cleaned analysis set for the charts and summary statistics.

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd

csv_candidates = [
    Path('wind_turbine_telemetry.csv'),
    Path('..') / 'wind_turbine_telemetry.csv',
    Path.cwd() / 'wind_turbine_telemetry.csv',
]

for candidate in csv_candidates:
    if candidate.exists():
        csv_path = candidate
        break
else:
    raise FileNotFoundError('Could not find wind_turbine_telemetry.csv. Run the mock data script first.')

df = pd.read_csv(csv_path, parse_dates=['Timestamp'])
df.info()
#df.describe(numeric_only=True)

<class 'pandas.DataFrame'>
RangeIndex: 6480 entries, 0 to 6479
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Timestamp              6480 non-null   datetime64[us]
 1   Turbine_ID             6480 non-null   str           
 2   Bearing_Temperature_C  6480 non-null   float64       
 3   Vibration_mm_s         5064 non-null   float64       
 4   RPM                    6480 non-null   float64       
 5   Grid_Priority          6480 non-null   str           
dtypes: datetime64[us](1), float64(3), str(2)
memory usage: 303.9 KB


In [6]:
df['Failure_State'] = np.where(
    df[['Bearing_Temperature_C', 'Vibration_mm_s']].isna().any(axis=1),
    'Sensor dropout / failure',
    'Normal',
)

failure_records = df[df['Failure_State'] == 'Sensor dropout / failure'].copy()
analysis_df = df.dropna(subset=['Bearing_Temperature_C', 'Vibration_mm_s']).copy()

print('Failure records retained for forensic review:', len(failure_records))
print('Rows available for statistical analysis:', len(analysis_df))
analysis_df.head()

Failure records retained for forensic review: 1416
Rows available for statistical analysis: 5064


,Timestamp,Turbine_ID,Bearing_Temperature_C,Vibration_mm_s,RPM,Grid_Priority,Failure_State
0,2026-01-01 00:00:00,WT-01,52.74,1.95,14.47,Standard,Normal
1,2026-01-01 01:00:00,WT-01,49.39,1.77,13.65,Standard,Normal
2,2026-01-01 02:00:00,WT-01,53.59,1.89,13.86,Standard,Normal
3,2026-01-01 03:00:00,WT-01,51.97,1.93,12.18,Standard,Normal
4,2026-01-01 04:00:00,WT-01,45.94,2.38,12.60,Standard,Normal


## Section 2: Interactive Anomaly Hunting via Scatter Plots [15 mins]Temperature, vibration, and RPM often move together in ways that say something about asset health. Interactive plots make it easier to inspect the relationships and spot when a turbine starts to drift away from normal behaviour.Use hover and zoom to inspect the point where behaviour changes.

In [9]:
import plotly.express as px

scatter_df = df.dropna(subset=['Bearing_Temperature_C', 'Vibration_mm_s', 'RPM']).copy()

fig = px.scatter(
    scatter_df,
    x='Bearing_Temperature_C',
    y='Vibration_mm_s',
    color='Turbine_ID',
    size='RPM',
    size_max=6,
    hover_data=['Timestamp', 'RPM', 'Grid_Priority'],
    title='Vibration vs Bearing Temperature by Turbine',
)
fig.update_layout(template='plotly_white')
fig

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': array([[Timestamp('2026-01-01 00:00:00'), 14.47, 'Standard'],
                                   [Timestamp('2026-01-01 01:00:00'), 13.65, 'Standard'],
                                   [Timestamp('2026-01-01 02:00:00'), 13.86, 'Standard'],
                                   ...,
                                   [Timestamp('2026-03-31 21:00:00'), 13.27, 'Standard'],
                                   [Timestamp('2026-03-31 22:00:00'), 12.74, 'Standard'],
                                   [Timestamp('2026-03-31 23:00:00'), 13.04, 'Standard']],
                                  shape=(2160, 3), dtype=object),
              'hovertemplate': ('Turbine_ID=WT-01<br>Bearing_Te' ... '{customdata[2]}<extra></extra>'),
              'legendgroup': 'WT-01',
              'marker': {'color': '#636efa',
                         'size': {'bdata': ('cT0K16PwLEDNzMzMzEwrQLgehetRuC' ... 'ejcD2KKkB7FK5H4XopQBSuR+F6FCpA'),
                                  'dtype': 'f8'},
                         'sizemode': 'area',
                         'sizeref': np.float64(0.4166666666666667),
                         'symbol': 'circle'},
              'mode': 'markers',
              'name': 'WT-01',
              'showlegend': True,
              'type': 'scattergl',
              'x': {'bdata': ('H4XrUbheSkBSuB6F67FIQOxRuB6Fy0' ... 'oUrkfhR0BSuB6F65FIQPYoXI/CtUZA'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('MzMzMzMz/z9SuB6F61H8Pz0K16NwPf' ... 'gehetRAkDNzMzMzMwCQGZmZmZmZv4/'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'customdata': array([[Timestamp('2026-01-01 00:00:00'), 13.66, 'Standard'],
                                   [Timestamp('2026-01-01 01:00:00'), 14.2, 'Standard'],
                                   [Timestamp('2026-01-01 02:00:00'), 13.65, 'Standard'],
                                   ...,
                                   [Timestamp('2026-01-31 21:00:00'), 0.0, 'Standard'],
                                   [Timestamp('2026-01-31 22:00:00'), 0.0, 'Standard'],
                                   [Timestamp('2026-01-31 23:00:00'), 0.55, 'Standard']],
                                  shape=(744, 3), dtype=object),
              'hovertemplate': ('Turbine_ID=WT-02<br>Bearing_Te' ... '{customdata[2]}<extra></extra>'),
              'legendgroup': 'WT-02',
              'marker': {'color': '#EF553B',
                         'size': {'bdata': ('UrgehetRK0BmZmZmZmYsQM3MzMzMTC' ... 'AAAAAAAAAAAAAAAAAAAJqZmZmZmeE/'),
                                  'dtype': 'f8'},
                         'sizemode': 'area',
                         'sizeref': np.float64(0.4166666666666667),
                         'symbol': 'circle'},
              'mode': 'markers',
              'name': 'WT-02',
              'showlegend': True,
              'type': 'scattergl',
              'x': {'bdata': ('FK5H4Xr0SEDhehSuR6FIQB+F61G4Xk' ... 'oUrkeBV0AAAAAAAMBXQAAAAAAAwFdA'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('rkfhehSuAUDsUbgehev9Px+F61G4Hg' ... 'AAAAAAIkDsUbgeheshQHE9CtejcCFA'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'customdata': array([[Timestamp('2026-01-01 00:00:00'), 14.33, 'Critical Infrastructure'],
                                   [Timestamp('2026-01-01 01:00:00'), 12.65, 'Critical Infrastructure'],
                                   [Timestamp('2026-01-01 02:00:00'), 12.34, 'Critical Infrastructure'],
                                   ...,
                                   [Timestamp('2026-03-31 21:00:00'), 11.66, 'Critical Infrastructure'],
                                   [Timestamp('2026-03-31 22:00:00'), 11.94, 'Critical Infrastructure'],
                                   [Timestamp('2026-03-31 23:00:00'), 12.1, 'Critical Infrastructure']],
                                  shape=(2160, 3), dtype=object),
            

In [10]:
temperature_df = df.sort_values(['Turbine_ID', 'Timestamp']).copy()

fig = px.line(
    temperature_df,
    x='Timestamp',
    y='Bearing_Temperature_C',
    color='Turbine_ID',
    markers=True,
    title='Bearing Temperature Over the 3-Month Timeline',
    hover_data={'Vibration_mm_s': True, 'RPM': True, 'Grid_Priority': True},
)
fig.update_layout(template='plotly_white', xaxis_title='Timestamp', yaxis_title='Bearing Temperature (C)')
fig

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': array([[1.95, 14.47, 'Standard'],
                                   [1.77, 13.65, 'Standard'],
                                   [1.89, 13.86, 'Standard'],
                                   ...,
                                   [2.29, 13.27, 'Standard'],
                                   [2.35, 12.74, 'Standard'],
                                   [1.9, 13.04, 'Standard']], shape=(2160, 3), dtype=object),
              'hovertemplate': ('Turbine_ID=WT-01<br>Timestamp=' ... '{customdata[2]}<extra></extra>'),
              'legendgroup': 'WT-01',
              'line': {'color': '#636efa', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines+markers',
              'name': 'WT-01',
              'showlegend': True,
              'type': 'scattergl',
              'x': array(['2026-01-01T00:00:00.000000', '2026-01-01T01:00:00.000000',
                          '2026-01-01T02:00:00.000000', ..., '2026-03-31T21:00:00.000000',
                          '2026-03-31T22:00:00.000000', '2026-03-31T23:00:00.000000'],
                         shape=(2160,), dtype='datetime64[us]'),
              'xaxis': 'x',
              'y': {'bdata': ('H4XrUbheSkBSuB6F67FIQOxRuB6Fy0' ... 'oUrkfhR0BSuB6F65FIQPYoXI/CtUZA'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'customdata': array([[2.21, 13.66, 'Standard'],
                                   [1.87, 14.2, 'Standard'],
                                   [2.14, 13.65, 'Standard'],
                                   ...,
                                   [nan, 0.0, 'Standard'],
                                   [nan, 0.0, 'Standard'],
                                   [nan, 0.0, 'Standard']], shape=(2160, 3), dtype=object),
              'hovertemplate': ('Turbine_ID=WT-02<br>Timestamp=' ... '{customdata[2]}<extra></extra>'),
              'legendgroup': 'WT-02',
              'line': {'color': '#EF553B', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines+markers',
              'name': 'WT-02',
              'showlegend': True,
              'type': 'scattergl',
              'x': array(['2026-01-01T00:00:00.000000', '2026-01-01T01:00:00.000000',
                          '2026-01-01T02:00:00.000000', ..., '2026-03-31T21:00:00.000000',
                          '2026-03-31T22:00:00.000000', '2026-03-31T23:00:00.000000'],
                         shape=(2160,), dtype='datetime64[us]'),
              'xaxis': 'x',
              'y': {'bdata': ('FK5H4Xr0SEDhehSuR6FIQB+F61G4Xk' ... 'AAAAAANkAAAAAAAAA2QAAAAAAAADZA'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'customdata': array([[2.0, 14.33, 'Critical Infrastructure'],
                                   [1.82, 12.65, 'Critical Infrastructure'],
                                   [2.39, 12.34, 'Critical Infrastructure'],
                                   ...,
                                   [6.0, 11.66, 'Critical Infrastructure'],
                                   [6.0, 11.94, 'Critical Infrastructure'],
                                   [5.9, 12.1, 'Critical Infrastructure']], shape=(2160, 3), dtype=object),
              'hovertemplate': ('Turbine_ID=WT-03<br>Timestamp=' ... '{customdata[2]}<extra></extra>'),
              'legendgroup': 'WT-03',
              'line': {'color': '#00cc96', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines+markers',
              'name': 'WT-03',
              'showlegend': True,
              'type': 'scattergl',
              'x': array(['2026-01-01T00:00:00.000000', '2026-01-01T01:00:00.000000',
                          '2026-01-01T02:00:00.000000', ..., '2026-03-31T21:00:00.000000',
                          '2026-03-31T22:00:00.000000', '2026-03-31T23:00:00.000000'],
                         shape=(2160,), dtype='datetime64[us]'),
              'xaxis': 'x',
             

### Interactive ChallengeUse Plotly's zoom and hover tools to find when WT-02 stops producing useful telemetry and when WT-03 starts to move away from the healthy pattern. The point is to connect the chart back to the asset, not just describe its shape.

## Section 3: Normalized Engineering Analysis [15 mins]Temperature, vibration, and RPM live on different scales, so direct comparison is hard. Normalisation puts them on a common scale and makes the drift easier to compare.This is useful when you want to compare trends or build a simple health score.

In [11]:
wt03_df = df[df['Turbine_ID'] == 'WT-03'].sort_values('Timestamp').copy()
features = ['Bearing_Temperature_C', 'Vibration_mm_s', 'RPM']

for feature in features:
    feature_min = wt03_df[feature].min()
    feature_max = wt03_df[feature].max()
    wt03_df[f'{feature}_Normalized'] = (wt03_df[feature] - feature_min) / (feature_max - feature_min)

normalized_long = wt03_df.melt(
    id_vars='Timestamp',
    value_vars=[f'{feature}_Normalized' for feature in features],
    var_name='Feature',
    value_name='Normalized_Value',
)

normalized_long['Feature'] = normalized_long['Feature'].str.replace('_Normalized', '', regex=False)

fig = px.line(
    normalized_long,
    x='Timestamp',
    y='Normalized_Value',
    color='Feature',
    markers=False,
    title='Normalized WT-03 Engineering Features',
)
fig.update_layout(template='plotly_white', yaxis_range=[0, 1])
fig

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': ('Feature=Bearing_Temperature_C<' ... 'ized_Value=%{y}<extra></extra>'),
              'legendgroup': 'Bearing_Temperature_C',
              'line': {'color': '#636efa', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines',
              'name': 'Bearing_Temperature_C',
              'showlegend': True,
              'type': 'scattergl',
              'x': array(['2026-01-01T00:00:00.000000', '2026-01-01T01:00:00.000000',
                          '2026-01-01T02:00:00.000000', ..., '2026-03-31T21:00:00.000000',
                          '2026-03-31T22:00:00.000000', '2026-03-31T23:00:00.000000'],
                         shape=(2160,), dtype='datetime64[us]'),
              'xaxis': 'x',
              'y': {'bdata': ('vUEOhhXToj+8AoM5+2awP0vdsbLunb' ... 'AAAAAA8D8AAAAAAADwPwAAAAAAAPA/'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': 'Feature=Vibration_mm_s<br>Timestamp=%{x}<br>Normalized_Value=%{y}<extra></extra>',
              'legendgroup': 'Vibration_mm_s',
              'line': {'color': '#EF553B', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines',
              'name': 'Vibration_mm_s',
              'showlegend': True,
              'type': 'scattergl',
              'x': array(['2026-01-01T00:00:00.000000', '2026-01-01T01:00:00.000000',
                          '2026-01-01T02:00:00.000000', ..., '2026-03-31T21:00:00.000000',
                          '2026-03-31T22:00:00.000000', '2026-03-31T23:00:00.000000'],
                         shape=(2160,), dtype='datetime64[us]'),
              'xaxis': 'x',
              'y': {'bdata': ('HMdxHMdxvD/gvJp4VjSyP6cutz/IUM' ... 'AAAAAA8D8AAAAAAADwP6D0SZ/0Se8/'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': 'Feature=RPM<br>Timestamp=%{x}<br>Normalized_Value=%{y}<extra></extra>',
              'legendgroup': 'RPM',
              'line': {'color': '#00cc96', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines',
              'name': 'RPM',
              'showlegend': True,
              'type': 'scattergl',
              'x': array(['2026-01-01T00:00:00.000000', '2026-01-01T01:00:00.000000',
                          '2026-01-01T02:00:00.000000', ..., '2026-03-31T21:00:00.000000',
                          '2026-03-31T22:00:00.000000', '2026-03-31T23:00:00.000000'],
                         shape=(2160,), dtype='datetime64[us]'),
              'xaxis': 'x',
              'y': {'bdata': ('3RAD0rGU6T9ayiTYVfjSP0trtk9UD8' ... 'AAAAAAAAAzOpYyCXa1Py0JdhW+3MA/'),
                    'dtype': 'f8'},
              'yaxis': 'y'}],
    'layout': {'legend': {'title': {'text': 'Feature'}, 'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Normalized WT-03 Engineering Features'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Timestamp'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'range': [0, 1], 'title': {'text': 'Normalized_Value'}}}
})

## Section 4: The Operational Decision & Dashboard Ethics [15 mins]Dashboards are not neutral. They influence maintenance timing, budget decisions, and safety risk. Use the chart to show which turbine should be prioritised if only one overhaul can be funded this quarter.WT-01 is healthy, WT-02 has already failed, and WT-03 is the one still moving in the wrong direction. The chart should make the trade-off obvious.

In [12]:
latest_rows = df.sort_values('Timestamp').groupby('Turbine_ID').tail(1).copy()

remaining_lifespan = {
    'WT-01': 120,
    'WT-02': 0,
    'WT-03': 14,
}

latest_rows['Remaining_Operational_Days'] = latest_rows['Turbine_ID'].map(remaining_lifespan)
latest_rows['Priority_Label'] = latest_rows['Grid_Priority']

fig = px.bar(
    latest_rows,
    x='Turbine_ID',
    y='Remaining_Operational_Days',
    color='Priority_Label',
    text='Remaining_Operational_Days',
    hover_data=['Bearing_Temperature_C', 'Vibration_mm_s', 'RPM'],
    title='Executive Dashboard: Remaining Operational Lifespan by Turbine',
)
fig.update_layout(template='plotly_white', yaxis_title='Estimated Remaining Operational Days')
fig

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': {'bdata': 'AAAAAAAANkAAAAAAAAD4fwAAAAAAAAAA9ihcj8K1RkBmZmZmZmb+PxSuR+F6FCpA',
                             'dtype': 'f8',
                             'shape': '2, 3'},
              'hovertemplate': ('Priority_Label=Standard<br>Tur' ... '{customdata[2]}<extra></extra>'),
              'legendgroup': 'Standard',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': 'Standard',
              'orientation': 'v',
              'showlegend': True,
              'text': {'bdata': 'AAAAAAAAAAAAAAAAAABeQA==', 'dtype': 'f8'},
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['WT-02', 'WT-01'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'AHg=', 'dtype': 'i1'},
              'yaxis': 'y'},
             {'customdata': {'bdata': 'AAAAAACAVECamZmZmZkXQDMzMzMzMyhA', 'dtype': 'f8', 'shape': '1, 3'},
              'hovertemplate': ('Priority_Label=Critical Infras' ... '{customdata[2]}<extra></extra>'),
              'legendgroup': 'Critical Infrastructure',
              'marker': {'color': '#EF553B', 'pattern': {'shape': ''}},
              'name': 'Critical Infrastructure',
              'orientation': 'v',
              'showlegend': True,
              'text': {'bdata': 'AAAAAAAALEA=', 'dtype': 'f8'},
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['WT-03'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'Dg==', 'dtype': 'i1'},
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'title': {'text': 'Priority_Label'}, 'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Executive Dashboard: Remaining Operational Lifespan by Turbine'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Turbine_ID'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'Estimated Remaining Operational Days'}}}
})

### Ethics & Governance ChallengeConsider the full cost of the decision. Do you spend the budget replacing WT-02, which is already offline and represents lost revenue, or do you intervene on WT-03 before it fails in the same way?Your answer should cover safety, environmental impact, operations, and liability. Data quality, visualisation design, and maintenance policy all feed into that choice.

## Wrap-UpBy the end of this workbook, you should be able to move from raw telemetry to a defensible operational recommendation using cleaning, plotting, normalisation, and a simple decision dashboard.